# Lumen AI — Diagnostic Channel Contribution Notebook

**Purpose:** Generate polished, VP-ready diagnostic contribution tables and plots from customer-level or aggregated attribution outputs.

This notebook is intentionally executive-friendly:
- Final plot labels use **Channel contribution** / **Adjusted contribution** / **Model contribution**.
- Technical terms such as SHAP and doubly robust are kept in the code/config where needed, but do not have to appear in final chart titles.
- The goal is to show what we learned from historical journeys before deciding where TD can act.

Supported input patterns:
1. **Customer-level attribution table:** one row per customer with paired columns like `dr_email_click` and `shap_email_click`.
2. **Aggregated attribution table:** one row per channel-interaction with columns like `interaction_name`, `mean_dr`, `mean_shap`.

Update Cell 1 to match your current table schema.


In [ ]:
# ===== 1) CONFIG =====

# Source table/dataframe. If a Python variable with this name exists, the notebook uses it.
# Otherwise it tries spark.table(TABLE_NAME).
TABLE_NAME = "lumen_ai_diagnostic_attribution_table"

CUSTOMER_ID_COL = "cost_ID"
CONVERSION_COL = "Converted"  # optional; update if available

# Choose input mode:
#   "customer_level" = one row per customer with paired attribution columns
#   "aggregated"     = one row per interaction with mean contribution columns
INPUT_MODE = "customer_level"

# Customer-level attribution column prefixes.
# Example columns: dr_email_click, shap_email_click
ADJUSTED_PREFIX = "dr_"
MODEL_PREFIX = "shap_"

# Aggregated table column names.
INTERACTION_COL = "interaction_name"
ADJUSTED_COL = "mean_dr"
MODEL_COL = "mean_shap"

# If your aggregated table uses different names, common alternatives are handled in Cell 3,
# but setting them here explicitly is best.

# Optional exposure / reach columns. Used only if present.
EXPOSED_CUSTOMERS_COL = "exposed_customers"
CONVERSION_RATE_COL = "conversion_rate"

# Channel parsing.
# Default: channel is inferred from the first token(s) of the interaction name.
# Use overrides when automatic parsing is wrong.
CHANNEL_NAME_OVERRIDES = {
    # "paid_search_click": "paid_search",
    # "direct_mail_contacted": "direct_mail",
}

# Known channel aliases and display order.
CHANNEL_ORDER = [
    "display",
    "seo",
    "paid_search",
    "leads",
    "direct_mail",
    "branch",
    "web",
    "email",
    "mobile",
    "phone",
]

CHANNEL_DISPLAY_NAMES = {
    "display": "Display",
    "seo": "SEO",
    "paid_search": "Paid Search",
    "leads": "Leads",
    "direct_mail": "Direct Mail",
    "branch": "Branch",
    "web": "Web",
    "email": "Email",
    "mobile": "Mobile",
    "phone": "Phone",
}

# Executive labels for plots/tables.
# Change these if you want simpler labels.
ADJUSTED_LABEL = "Adjusted contribution"
MODEL_LABEL = "Model contribution"
GAP_LABEL = "Adjusted − model"

# Channel aggregation method:
#   "mean" = mean attribution across interaction types within channel, like your screenshot
#   "sum"  = total channel contribution across interaction types
CHANNEL_AGG_METHOD = "mean"

# Plot/export options.
OUTPUT_DIR = "/dbfs/FileStore/lumen_ai_diagnostic"
TOP_N_INTERACTIONS = 18
SHOW_MODEL_COMPARISON = True  # False = final plots show adjusted contribution only

# TD-style plot config.
TD_GREEN_DARK = "#005D3B"
TD_GREEN = "#008A00"
TD_GREEN_MED = "#2E7D32"
TD_GREEN_LIGHT = "#A5D6A7"
TD_GREEN_PALE = "#E8F5E9"
TD_GOLD = "#B8860B"
TD_GRAY = "#5F6368"

PLOT_SIZES = {
    "channel_group": (12, 7),
    "interaction": (12, 8),
    "gap": (11, 7),
    "scatter": (7, 7),
    "summary_table": (12, 5),
}

PLOT_CONFIG = {
    "font_size": 11,
    "title_size": 15,
    "label_size": 11,
    "annotation_size": 10,
    "dpi": 220,
    "value_label_decimals": 4,
    "zero_line_width": 1.0,
}


In [ ]:
# ===== 2) Imports, plotting style, and load table =====

from pyspark.sql import functions as F
from pyspark.sql import Window
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re
from functools import reduce

if OUTPUT_DIR:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({
    "font.size": PLOT_CONFIG["font_size"],
    "axes.titlesize": PLOT_CONFIG["title_size"],
    "axes.labelsize": PLOT_CONFIG["label_size"],
    "xtick.labelsize": PLOT_CONFIG["font_size"],
    "ytick.labelsize": PLOT_CONFIG["font_size"],
    "legend.fontsize": PLOT_CONFIG["font_size"],
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#333333",
    "axes.grid": True,
    "grid.alpha": 0.25,
})

def ddisplay(obj):
    try:
        display(obj)
    except Exception:
        print(obj)

def save_plot(name):
    if OUTPUT_DIR:
        path = f"{OUTPUT_DIR}/{name}.png"
        plt.savefig(path, dpi=PLOT_CONFIG["dpi"], bbox_inches="tight")
        print(f"Saved: {path}")

if TABLE_NAME in globals():
    raw_df = globals()[TABLE_NAME]
else:
    raw_df = spark.table(TABLE_NAME)

print(f"Loaded table/dataframe: {TABLE_NAME}")
print(f"Rows: {raw_df.count():,}")
print(f"Columns: {len(raw_df.columns):,}")


In [ ]:
# ===== 3) Helper functions =====

def standardize_interaction_name(name: str) -> str:
    name = str(name)
    for prefix in [ADJUSTED_PREFIX, MODEL_PREFIX, "channel_"]:
        if name.startswith(prefix):
            name = name[len(prefix):]
    return name.lower().strip()

def infer_channel_from_interaction(interaction: str) -> str:
    interaction = standardize_interaction_name(interaction)
    if interaction in CHANNEL_NAME_OVERRIDES:
        return CHANNEL_NAME_OVERRIDES[interaction]

    # specific multi-token channels first
    if interaction.startswith("paid_search") or interaction.startswith("paidsearch") or interaction.startswith("paid_"):
        return "paid_search"
    if interaction.startswith("direct_mail") or interaction.startswith("directmail") or interaction.startswith("direct_"):
        return "direct_mail"

    first = interaction.split("_")[0]
    if first in CHANNEL_ORDER:
        return first

    # fallback: first token
    return first

def pretty_interaction_label(interaction: str) -> str:
    x = standardize_interaction_name(interaction)
    x = x.replace("paid_search", "paid search").replace("direct_mail", "direct mail")
    return x.replace("_", " ")

def pretty_channel_label(channel: str) -> str:
    return CHANNEL_DISPLAY_NAMES.get(channel, str(channel).replace("_", " ").title())

def find_paired_attribution_columns(df):
    cols = df.columns
    adjusted_suffixes = {
        c[len(ADJUSTED_PREFIX):]: c
        for c in cols
        if c.startswith(ADJUSTED_PREFIX)
    }
    model_suffixes = {
        c[len(MODEL_PREFIX):]: c
        for c in cols
        if c.startswith(MODEL_PREFIX)
    }
    suffixes = sorted(set(adjusted_suffixes.keys()).intersection(model_suffixes.keys()))
    pairs = [(suffix, adjusted_suffixes[suffix], model_suffixes[suffix]) for suffix in suffixes]
    return pairs

def build_customer_level_summary(df):
    pairs = find_paired_attribution_columns(df)
    if not pairs:
        raise ValueError(
            f"No paired attribution columns found using ADJUSTED_PREFIX={ADJUSTED_PREFIX} and MODEL_PREFIX={MODEL_PREFIX}."
        )

    agg_exprs = []
    for suffix, adj_col, mod_col in pairs:
        safe_suffix = re.sub(r"[^A-Za-z0-9_]+", "_", suffix)
        agg_exprs += [
            F.avg(F.col(adj_col).cast("double")).alias(f"{safe_suffix}__adjusted_mean"),
            F.avg(F.col(mod_col).cast("double")).alias(f"{safe_suffix}__model_mean"),
            F.count(F.when(F.col(adj_col).isNotNull(), 1)).alias(f"{safe_suffix}__adjusted_non_null"),
            F.count(F.when(F.col(mod_col).isNotNull(), 1)).alias(f"{safe_suffix}__model_non_null"),
        ]

    row = df.agg(*agg_exprs).first().asDict()
    records = []
    for suffix, adj_col, mod_col in pairs:
        safe_suffix = re.sub(r"[^A-Za-z0-9_]+", "_", suffix)
        interaction = standardize_interaction_name(suffix)
        channel = infer_channel_from_interaction(interaction)
        records.append({
            "interaction_name": interaction,
            "interaction_label": pretty_interaction_label(interaction),
            "channel": channel,
            "channel_label": pretty_channel_label(channel),
            "mean_adjusted": row.get(f"{safe_suffix}__adjusted_mean"),
            "mean_model": row.get(f"{safe_suffix}__model_mean"),
            "adjusted_non_null": row.get(f"{safe_suffix}__adjusted_non_null"),
            "model_non_null": row.get(f"{safe_suffix}__model_non_null"),
        })

    pdf = pd.DataFrame(records)
    pdf["adjusted_minus_model"] = pdf["mean_adjusted"] - pdf["mean_model"]
    pdf["abs_gap"] = pdf["adjusted_minus_model"].abs()
    return pdf

def infer_aggregated_cols(df):
    cols = df.columns
    interaction_col = INTERACTION_COL if INTERACTION_COL in cols else None
    adjusted_col = ADJUSTED_COL if ADJUSTED_COL in cols else None
    model_col = MODEL_COL if MODEL_COL in cols else None

    # common alternatives
    for cand in ["interaction_name", "Channel × interaction", "channel_interaction", "feature_name"]:
        if interaction_col is None and cand in cols:
            interaction_col = cand
    for cand in ["mean_dr", "Mean DR", "dr_value", "Mean adjusted", "mean_adjusted"]:
        if adjusted_col is None and cand in cols:
            adjusted_col = cand
    for cand in ["mean_shap", "Mean SHAP", "shap_value", "Mean model", "mean_model"]:
        if model_col is None and cand in cols:
            model_col = cand

    if interaction_col is None or adjusted_col is None or model_col is None:
        raise ValueError(
            f"Could not infer aggregated attribution columns. Found columns: {cols}"
        )
    return interaction_col, adjusted_col, model_col

def build_aggregated_summary(df):
    interaction_col, adjusted_col, model_col = infer_aggregated_cols(df)
    pdf = (
        df.select(
            F.col(interaction_col).cast("string").alias("interaction_name_raw"),
            F.col(adjusted_col).cast("double").alias("mean_adjusted"),
            F.col(model_col).cast("double").alias("mean_model"),
        )
        .toPandas()
    )
    pdf["interaction_name"] = pdf["interaction_name_raw"].map(standardize_interaction_name)
    pdf["interaction_label"] = pdf["interaction_name"].map(pretty_interaction_label)
    pdf["channel"] = pdf["interaction_name"].map(infer_channel_from_interaction)
    pdf["channel_label"] = pdf["channel"].map(pretty_channel_label)
    pdf["adjusted_minus_model"] = pdf["mean_adjusted"] - pdf["mean_model"]
    pdf["abs_gap"] = pdf["adjusted_minus_model"].abs()
    return pdf

def build_channel_summary(interaction_pdf):
    if CHANNEL_AGG_METHOD == "sum":
        grouped = interaction_pdf.groupby(["channel", "channel_label"], as_index=False).agg(
            n_interactions=("interaction_name", "count"),
            mean_adjusted=("mean_adjusted", "sum"),
            mean_model=("mean_model", "sum"),
        )
    else:
        grouped = interaction_pdf.groupby(["channel", "channel_label"], as_index=False).agg(
            n_interactions=("interaction_name", "count"),
            mean_adjusted=("mean_adjusted", "mean"),
            mean_model=("mean_model", "mean"),
        )
    grouped["adjusted_minus_model"] = grouped["mean_adjusted"] - grouped["mean_model"]
    grouped["abs_gap"] = grouped["adjusted_minus_model"].abs()

    # Sort by adjusted contribution descending by default.
    grouped["channel_order"] = grouped["channel"].apply(lambda x: CHANNEL_ORDER.index(x) if x in CHANNEL_ORDER else 999)
    grouped = grouped.sort_values("mean_adjusted", ascending=True).reset_index(drop=True)
    return grouped

def add_value_labels_h(ax, values, x_offset=0.0003):
    dec = PLOT_CONFIG["value_label_decimals"]
    for y, val in enumerate(values):
        if pd.isna(val):
            continue
        ha = "left" if val >= 0 else "right"
        dx = x_offset if val >= 0 else -x_offset
        ax.text(val + dx, y, f"{val:+.{dec}f}", va="center", ha=ha, fontsize=PLOT_CONFIG["annotation_size"])

def plot_channel_contribution(channel_pdf, title=None, show_model=True):
    plot_pdf = channel_pdf.copy().sort_values("mean_adjusted", ascending=True)
    y = np.arange(len(plot_pdf))
    fig, ax = plt.subplots(figsize=PLOT_SIZES["channel_group"])

    if show_model:
        h = 0.38
        ax.barh(y - h/2, plot_pdf["mean_adjusted"], height=h, color=TD_GREEN_DARK, label=ADJUSTED_LABEL)
        ax.barh(y + h/2, plot_pdf["mean_model"], height=h, color=TD_GOLD, label=MODEL_LABEL)
        # labels for adjusted only to reduce clutter
        for idx, val in enumerate(plot_pdf["mean_adjusted"]):
            ha = "left" if val >= 0 else "right"
            dx = 0.0003 if val >= 0 else -0.0003
            ax.text(val + dx, idx - h/2, f"{val:+.4f}", va="center", ha=ha, fontsize=9)
    else:
        ax.barh(y, plot_pdf["mean_adjusted"], color=TD_GREEN_DARK, label=ADJUSTED_LABEL)
        add_value_labels_h(ax, plot_pdf["mean_adjusted"].values)

    labels = [f"{row.channel_label}  (n={int(row.n_interactions)})" for _, row in plot_pdf.iterrows()]
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.axvline(0, color="#333333", linewidth=PLOT_CONFIG["zero_line_width"])
    ax.set_xlabel("Mean contribution across interaction types within channel" if CHANNEL_AGG_METHOD == "mean" else "Total contribution across interaction types within channel")
    ax.set_title(title or "Channel contribution summary", fontweight="bold", color=TD_GREEN_DARK, pad=14)
    if show_model:
        ax.legend(loc="lower right")
    plt.tight_layout()
    return fig, ax

def plot_interaction_contribution(interaction_pdf, title=None, top_n=TOP_N_INTERACTIONS, show_model=True):
    plot_pdf = interaction_pdf.dropna(subset=["mean_adjusted"]).copy()
    plot_pdf = plot_pdf.sort_values("mean_adjusted", ascending=False).head(top_n).sort_values("mean_adjusted", ascending=True)
    y = np.arange(len(plot_pdf))
    fig, ax = plt.subplots(figsize=PLOT_SIZES["interaction"])

    if show_model:
        h = 0.38
        ax.barh(y - h/2, plot_pdf["mean_adjusted"], height=h, color=TD_GREEN_DARK, label=ADJUSTED_LABEL)
        ax.barh(y + h/2, plot_pdf["mean_model"], height=h, color=TD_GOLD, label=MODEL_LABEL)
        for idx, val in enumerate(plot_pdf["mean_adjusted"]):
            ha = "left" if val >= 0 else "right"
            dx = 0.0003 if val >= 0 else -0.0003
            ax.text(val + dx, idx - h/2, f"{val:+.4f}", va="center", ha=ha, fontsize=9)
    else:
        ax.barh(y, plot_pdf["mean_adjusted"], color=TD_GREEN_DARK)
        add_value_labels_h(ax, plot_pdf["mean_adjusted"].values)

    ax.set_yticks(y)
    ax.set_yticklabels(plot_pdf["interaction_label"])
    ax.axvline(0, color="#333333", linewidth=PLOT_CONFIG["zero_line_width"])
    ax.set_xlabel("Mean contribution")
    ax.set_title(title or f"Top {top_n} channel-interaction contributions", fontweight="bold", color=TD_GREEN_DARK, pad=14)
    if show_model:
        ax.legend(loc="lower right")
    plt.tight_layout()
    return fig, ax

def plot_gap(interaction_pdf, level="interaction"):
    if level == "channel":
        pdf = build_channel_summary(interaction_pdf)
        label_col = "channel_label"
        size = PLOT_SIZES["gap"]
        title = "Where adjusted contribution differs most from model contribution — channel level"
    else:
        pdf = interaction_pdf.copy()
        label_col = "interaction_label"
        size = PLOT_SIZES["gap"]
        title = "Where adjusted contribution differs most from model contribution — interaction level"

    plot_pdf = pdf.sort_values("abs_gap", ascending=False).head(TOP_N_INTERACTIONS).sort_values("adjusted_minus_model", ascending=True)
    colors = [TD_GREEN_DARK if v >= 0 else TD_GOLD for v in plot_pdf["adjusted_minus_model"]]
    fig, ax = plt.subplots(figsize=size)
    ax.barh(plot_pdf[label_col], plot_pdf["adjusted_minus_model"], color=colors)
    ax.axvline(0, color="#333333", linewidth=PLOT_CONFIG["zero_line_width"])
    ax.set_xlabel(GAP_LABEL)
    ax.set_title(title, fontweight="bold", color=TD_GREEN_DARK, pad=14)
    add_value_labels_h(ax, plot_pdf["adjusted_minus_model"].values)
    plt.tight_layout()
    return fig, ax


In [ ]:
# ===== 4) Build standardized interaction-level and channel-level summaries =====

if INPUT_MODE == "customer_level":
    interaction_pdf = build_customer_level_summary(raw_df)
elif INPUT_MODE == "aggregated":
    interaction_pdf = build_aggregated_summary(raw_df)
else:
    raise ValueError("INPUT_MODE must be either 'customer_level' or 'aggregated'.")

# Standard sorting.
interaction_pdf = interaction_pdf.sort_values("mean_adjusted", ascending=False).reset_index(drop=True)
channel_pdf = build_channel_summary(interaction_pdf)

# Display clean tables.
interaction_display = interaction_pdf[[
    "interaction_label",
    "channel_label",
    "mean_adjusted",
    "mean_model",
    "adjusted_minus_model",
    "abs_gap",
]].rename(columns={
    "interaction_label": "Channel × interaction",
    "channel_label": "Channel",
    "mean_adjusted": ADJUSTED_LABEL,
    "mean_model": MODEL_LABEL,
    "adjusted_minus_model": GAP_LABEL,
    "abs_gap": "Absolute gap",
})

channel_display = channel_pdf[[
    "channel_label",
    "n_interactions",
    "mean_adjusted",
    "mean_model",
    "adjusted_minus_model",
]].rename(columns={
    "channel_label": "Channel",
    "n_interactions": "n interaction types",
    "mean_adjusted": ADJUSTED_LABEL,
    "mean_model": MODEL_LABEL,
    "adjusted_minus_model": GAP_LABEL,
})

print("Interaction-level contribution summary")
ddisplay(spark.createDataFrame(interaction_display))

print("Channel-level contribution summary")
ddisplay(spark.createDataFrame(channel_display))


In [ ]:
# ===== 5) Final diagnostic plot: channel contribution summary =====
# This is the main executive-friendly diagnostic plot.

fig, ax = plot_channel_contribution(
    channel_pdf,
    title="Channel contribution summary",
    show_model=SHOW_MODEL_COMPARISON,
)
save_plot("diagnostic_channel_contribution_summary")
plt.show()

print("Message:")
print("- This plot summarizes which channels appear to contribute most after accounting for customer context and treatment-selection patterns.")
print("- Channels with positive adjusted contribution are directionally favorable; negative values may indicate weak contribution, adverse reaction patterns, or model/overlap issues to investigate.")


In [ ]:
# ===== 6) Channel-level adjusted-only version =====
# Use this if you want a simpler VP-ready plot with no method comparison.

fig, ax = plot_channel_contribution(
    channel_pdf,
    title="Adjusted channel contribution",
    show_model=False,
)
save_plot("diagnostic_adjusted_channel_contribution_only")
plt.show()


In [ ]:
# ===== 7) Interaction-level contribution plot =====
# More granular appendix view.

fig, ax = plot_interaction_contribution(
    interaction_pdf,
    title=f"Top {TOP_N_INTERACTIONS} channel-interaction contributions",
    top_n=TOP_N_INTERACTIONS,
    show_model=SHOW_MODEL_COMPARISON,
)
save_plot("diagnostic_top_interaction_contributions")
plt.show()


In [ ]:
# ===== 8) Contribution gap plots =====
# Useful for appendix: where adjusted contribution and model contribution disagree most.

fig, ax = plot_gap(interaction_pdf, level="interaction")
save_plot("diagnostic_interaction_adjusted_minus_model_gap")
plt.show()

fig, ax = plot_gap(interaction_pdf, level="channel")
save_plot("diagnostic_channel_adjusted_minus_model_gap")
plt.show()

print("Message:")
print("- Large positive gaps mean adjusted contribution gives more credit than the model-contribution view.")
print("- Large negative gaps mean adjusted contribution gives less credit than the model-contribution view.")
print("- These are useful for diagnostics, but the VP slide can simply say 'channel contribution' without method terminology.")


In [ ]:
# ===== 9) Sign agreement and ranking diagnostics =====
# Appendix sanity check.

diagnostic_pdf = interaction_pdf.copy()
diagnostic_pdf["adjusted_sign"] = np.where(diagnostic_pdf["mean_adjusted"] > 0, "positive",
                                  np.where(diagnostic_pdf["mean_adjusted"] < 0, "negative", "zero"))
diagnostic_pdf["model_sign"] = np.where(diagnostic_pdf["mean_model"] > 0, "positive",
                               np.where(diagnostic_pdf["mean_model"] < 0, "negative", "zero"))
diagnostic_pdf["same_sign"] = diagnostic_pdf["adjusted_sign"] == diagnostic_pdf["model_sign"]

diagnostic_pdf["adjusted_rank"] = diagnostic_pdf["mean_adjusted"].rank(ascending=False, method="min").astype(int)
diagnostic_pdf["model_rank"] = diagnostic_pdf["mean_model"].rank(ascending=False, method="min").astype(int)
diagnostic_pdf["rank_gap"] = diagnostic_pdf["model_rank"] - diagnostic_pdf["adjusted_rank"]

sign_summary = pd.DataFrame([{
    "interactions": len(diagnostic_pdf),
    "positive_adjusted": int((diagnostic_pdf["mean_adjusted"] > 0).sum()),
    "negative_adjusted": int((diagnostic_pdf["mean_adjusted"] < 0).sum()),
    "positive_model": int((diagnostic_pdf["mean_model"] > 0).sum()),
    "negative_model": int((diagnostic_pdf["mean_model"] < 0).sum()),
    "same_sign_count": int(diagnostic_pdf["same_sign"].sum()),
    "same_sign_share": float(diagnostic_pdf["same_sign"].mean()) if len(diagnostic_pdf) else None,
}])

rank_display = diagnostic_pdf[[
    "interaction_label",
    "channel_label",
    "mean_adjusted",
    "mean_model",
    "adjusted_minus_model",
    "adjusted_sign",
    "model_sign",
    "same_sign",
    "adjusted_rank",
    "model_rank",
    "rank_gap",
]].rename(columns={
    "interaction_label": "Channel × interaction",
    "channel_label": "Channel",
    "mean_adjusted": ADJUSTED_LABEL,
    "mean_model": MODEL_LABEL,
    "adjusted_minus_model": GAP_LABEL,
})

print("Sign agreement summary")
ddisplay(spark.createDataFrame(sign_summary))

print("Interaction ranking diagnostic")
ddisplay(spark.createDataFrame(rank_display.sort_values("adjusted_rank")))


In [ ]:
# ===== 10) Copy-ready final numbers for PowerPoint =====

top_adjusted_channel = channel_pdf.sort_values("mean_adjusted", ascending=False).head(1)
bottom_adjusted_channel = channel_pdf.sort_values("mean_adjusted", ascending=True).head(1)
top_adjusted_interaction = interaction_pdf.sort_values("mean_adjusted", ascending=False).head(1)
bottom_adjusted_interaction = interaction_pdf.sort_values("mean_adjusted", ascending=True).head(1)
largest_gap = interaction_pdf.sort_values("abs_gap", ascending=False).head(1)

summary_lines = []

if not top_adjusted_channel.empty:
    r = top_adjusted_channel.iloc[0]
    summary_lines.append(f"Top adjusted channel contribution: {r['channel_label']} ({r['mean_adjusted']:+.4f}).")
if not bottom_adjusted_channel.empty:
    r = bottom_adjusted_channel.iloc[0]
    summary_lines.append(f"Lowest adjusted channel contribution: {r['channel_label']} ({r['mean_adjusted']:+.4f}).")
if not top_adjusted_interaction.empty:
    r = top_adjusted_interaction.iloc[0]
    summary_lines.append(f"Top adjusted interaction contribution: {r['interaction_label']} ({r['mean_adjusted']:+.4f}).")
if not bottom_adjusted_interaction.empty:
    r = bottom_adjusted_interaction.iloc[0]
    summary_lines.append(f"Lowest adjusted interaction contribution: {r['interaction_label']} ({r['mean_adjusted']:+.4f}).")
if not largest_gap.empty:
    r = largest_gap.iloc[0]
    summary_lines.append(f"Largest difference between adjusted and model contribution: {r['interaction_label']} ({r['adjusted_minus_model']:+.4f}).")

summary_lines.append(f"Number of channel groups analyzed: {channel_pdf['channel'].nunique():,}.")
summary_lines.append(f"Number of channel-interaction types analyzed: {len(interaction_pdf):,}.")
summary_lines.append(f"Interactions with positive adjusted contribution: {(interaction_pdf['mean_adjusted'] > 0).sum():,}.")
summary_lines.append(f"Interactions with negative adjusted contribution: {(interaction_pdf['mean_adjusted'] < 0).sum():,}.")

print("\nCOPY-READY SUMMARY")
print("==================")
for line in summary_lines:
    print("- " + line)

summary_table_pdf = pd.DataFrame({"summary": summary_lines})
ddisplay(spark.createDataFrame(summary_table_pdf))


## Interpretation guardrails

Use executive-friendly wording:
- “Channel contribution”
- “Adjusted contribution”
- “Historical learning”
- “Diagnostic lens”
- “Accounting for customer context and selection effects”

Avoid overclaiming:
- Do not say this proves causal lift.
- Do not say every diagnostic channel is a directly assignable future action.
- Paid search, SEO, display, and web can be useful diagnostic signals even if they are not clean customer-level outbound actions.
